# Error Functions in Machine Learning

An **error function** (also called a loss or cost function) tells us how far off our model's predictions are from the true answers. The goal of training is to **minimize** this error.

Think of it like a grade on a test — the error function scores how badly the model performed, and the model learns by trying to lower that score.

## Binary Classification Error

$$E = -\frac{1}{m} \sum_{i=1}^{m} (1 - y_i) \ln(1 - \hat{y}_i) + y_i \ln(\hat{y}_i)$$

This is the **cross-entropy loss** for binary classification (2 classes: 0 or 1).

| Symbol | Meaning |
|--------|---------|
| $m$ | Total number of training samples |
| $y_i$ | True label for sample $i$ (either 0 or 1) |
| $\hat{y}_i$ | Model's predicted probability that sample $i$ is class 1 |
| $\frac{1}{m}$ | Averages the error over all samples |

**How it works:** For each sample, only one term is "active":
- If $y_i = 1$: we compute $\ln(\hat{y}_i)$ → penalizes predictions far from 1
- If $y_i = 0$: we compute $\ln(1 - \hat{y}_i)$ → penalizes predictions far from 0

## Total Error as a Function of Weights and Bias

$$E(W, b) = -\frac{1}{m} \sum_{i=1}^{m} (1 - y_i) \ln(1 - \sigma(Wx^{(i)} + b)) + y_i \ln(\sigma(Wx^{(i)} + b))$$

This is the same error function, but written to show **what the model actually computes**:

| Symbol | Meaning |
|--------|---------|
| $W$ | Weights — how much each input feature matters |
| $b$ | Bias — a constant offset |
| $x^{(i)}$ | The input features for sample $i$ |
| $\sigma(\cdot)$ | The sigmoid function — squishes any number into the range (0, 1) |

The flow is: `inputs → linear combination (Wx + b) → sigmoid → probability → error`

This form is useful because **gradient descent** needs to know how the error depends on $W$ and $b$ to update them.

## Multiclass Error

$$E = -\frac{1}{m} \sum_{i=1}^{m} \sum_{j=1}^{n} y_{ij} \ln(\hat{y}_{ij})$$

When there are more than 2 classes (e.g., recognizing digits 0–9):

| Symbol | Meaning |
|--------|---------|
| $n$ | Number of classes |
| $y_{ij}$ | 1 if sample $i$ belongs to class $j$, else 0 (one-hot encoded) |
| $\hat{y}_{ij}$ | Model's predicted probability that sample $i$ is class $j$ |

The double sum goes over every sample AND every class. For each sample, only the correct class contributes (since $y_{ij} = 0$ for wrong classes).

## Derivative of the Sigmoid Function

$$\sigma'(x) = \sigma(x)(1 - \sigma(x))$$

This is the **derivative** of the sigmoid function, which is essential for **backpropagation** — the algorithm used to update weights during training.

| Symbol | Meaning |
|--------|---------|
| $\sigma(x)$ | The sigmoid function: $\sigma(x) = \frac{1}{1 + e^{-x}}$ |
| $\sigma'(x)$ | The rate of change of the sigmoid at point $x$ |

**Why it matters:** During gradient descent, we need to compute how the error changes with respect to the weights. Since the sigmoid is part of the model, its derivative appears in the chain rule calculations.

**Key property:** The derivative is expressed entirely in terms of the sigmoid itself, making it computationally efficient — no need to recalculate from scratch.

**Note:** The maximum value of $\sigma'(x)$ is $0.25$, occurring at $x = 0$. This means gradients can become very small for large $|x|$, which is known as the **vanishing gradient problem**.


## Gradient of the Error Function

$$\nabla E = \left( \frac{\partial E}{\partial w_1}, \cdots, \frac{\partial E}{\partial w_n}, \frac{\partial E}{\partial b} \right)$$

The **gradient** is a vector of all partial derivatives of the error function with respect to each parameter. It points in the direction of **steepest increase** of the error.

| Symbol | Meaning |
|--------|---------|
| $\nabla E$ | The gradient vector of the error function |
| $\frac{\partial E}{\partial w_j}$ | How much the error changes when weight $w_j$ changes slightly |
| $\frac{\partial E}{\partial b}$ | How much the error changes when the bias $b$ changes slightly |
| $n$ | Number of input features (and corresponding weights) |

**Why it matters:** In **gradient descent**, we update the weights and bias by moving in the **opposite direction** of the gradient to minimize the error:

$$w_j \leftarrow w_j - \alpha \frac{\partial E}{\partial w_j}, \quad b \leftarrow b - \alpha \frac{\partial E}{\partial b}$$

where $\alpha$ is the **learning rate** — a hyperparameter that controls the step size.

**Key insight:** The gradient tells the model *which direction* to adjust each parameter and *by how much*, making it the core mechanism behind learning.

## Gradient Calculation — Step-by-Step Derivation

Now that we know the gradient is the vector of partial derivatives, let's actually **compute** it step by step.

To simplify the calculation, we focus on the error produced by a **single point** first. The total error is just the average of all individual errors, so once we know the derivative for one point, we can average over all $m$ points.

The error produced by a single point is:

$$E = -y \ln(\hat{y}) - (1 - y) \ln(1 - \hat{y})$$

where the prediction is $\hat{y} = \sigma(Wx + b)$.

Our goal is to compute the partial derivatives $\frac{\partial E}{\partial w_j}$ and $\frac{\partial E}{\partial b}$.

### Step 1: Derivative of $\hat{y}$ with respect to $w_j$

Since $\hat{y} = \sigma(Wx + b)$, we apply the **chain rule**. Recall that $\sigma'(x) = \sigma(x)(1 - \sigma(x))$, so:

$$\frac{\partial}{\partial w_j} \hat{y} = \frac{\partial}{\partial w_j} \sigma(Wx + b) = \sigma'(Wx + b) \cdot \frac{\partial}{\partial w_j}(Wx + b)$$

Now, expanding $Wx + b = w_1 x_1 + w_2 x_2 + \cdots + w_n x_n + b$, the only term that depends on $w_j$ is $w_j x_j$, so:

$$\frac{\partial}{\partial w_j}(Wx + b) = x_j$$

Putting it together:

$$\frac{\partial \hat{y}}{\partial w_j} = \sigma'(Wx + b) \cdot x_j = \hat{y}(1 - \hat{y}) \cdot x_j$$

### Step 2: Derivative of $E$ with respect to $w_j$

Now we compute $\frac{\partial E}{\partial w_j}$ using the chain rule on $E = -y \ln(\hat{y}) - (1 - y) \ln(1 - \hat{y})$:

$$\frac{\partial E}{\partial w_j} = -y \cdot \frac{1}{\hat{y}} \cdot \frac{\partial \hat{y}}{\partial w_j} - (1 - y) \cdot \frac{1}{1 - \hat{y}} \cdot (-1) \cdot \frac{\partial \hat{y}}{\partial w_j}$$

Substituting $\frac{\partial \hat{y}}{\partial w_j} = \hat{y}(1 - \hat{y}) \cdot x_j$ from Step 1:

$$= -y \cdot \frac{1}{\hat{y}} \cdot \hat{y}(1 - \hat{y}) \cdot x_j + (1 - y) \cdot \frac{1}{1 - \hat{y}} \cdot \hat{y}(1 - \hat{y}) \cdot x_j$$

$$= -y(1 - \hat{y}) \cdot x_j + (1 - y)\hat{y} \cdot x_j$$

$$= \left[-y + y\hat{y} + \hat{y} - y\hat{y}\right] x_j$$

$$= (\hat{y} - y) \cdot x_j = -(y - \hat{y}) \cdot x_j$$

**Result:**

$$\frac{\partial E}{\partial w_j} = -(y - \hat{y}) \cdot x_j$$

### Step 3: Derivative of $E$ with respect to $b$

A similar calculation (replacing $w_j$ with $b$) gives us:

$$\frac{\partial E}{\partial b} = -(y - \hat{y})$$

This follows the same logic as Step 2, except that $\frac{\partial}{\partial b}(Wx + b) = 1$ instead of $x_j$.

### Step 4: The Full Gradient — A Beautiful Result

Combining the results from Steps 2 and 3, for a point with coordinates $(x_1, \ldots, x_n)$, label $y$, and prediction $\hat{y}$, the gradient of the error function at that point is:

$$\nabla E = \left(-(y - \hat{y})x_1, \;\cdots, \;-(y - \hat{y})x_n, \;-(y - \hat{y})\right)$$

This can be written compactly as:

$$\nabla E = -(y - \hat{y})(x_1, \ldots, x_n, 1)$$

**Why this is fascinating:** The gradient is a **scalar** times the coordinates of the point (with a 1 appended for the bias). And what is that scalar? It's $-(y - \hat{y})$ — a multiple of the **difference between the label and the prediction**.

| Scenario | $(y - \hat{y})$ | Effect |
|----------|-----------------|--------|
| Label is 1, prediction close to 0 | Large positive | Large gradient → big update toward the point |
| Label is 0, prediction close to 1 | Large negative | Large gradient → big update away from the point |
| Prediction is close to label | Near zero | Small gradient → tiny update (already doing well) |

**Intuition:** A misclassified point produces a large gradient, pushing the model to correct its mistake. A correctly classified point barely affects the weights. This is exactly the behavior we want from a learning algorithm.

### Gradient Descent Step

So, a small gradient means we'll change our coordinates by a little bit, and a large gradient means we'll change our coordinates by a lot. If this sounds anything like the perceptron algorithm, this is no coincidence!

Since the gradient descent step consists in **subtracting a multiple of the gradient** of the error function at every point, the weights update as:

$$w_i' \leftarrow w_i - \alpha \left[-(y - \hat{y})x_i\right]$$

which simplifies to:

$$w_i' \leftarrow w_i + \alpha (y - \hat{y}) x_i$$

Similarly, the bias updates as:

$$b' \leftarrow b + \alpha (y - \hat{y})$$

| Symbol | Meaning |
|--------|---------|
| $w_i$ | Current weight for feature $i$ |
| $w_i'$ | Updated weight for feature $i$ after one gradient descent step |
| $\alpha$ | Learning rate — controls the step size of each update |
| $y$ | True label for the sample (0 or 1) |
| $\hat{y}$ | Model's predicted probability for the sample |
| $x_i$ | The $i$-th input feature of the sample |
| $b$ | Current bias |
| $b'$ | Updated bias after one gradient descent step |

**Note:** Since we've taken the average of the errors, the term we are adding should technically be $\frac{1}{m} \cdot \alpha$ instead of $\alpha$. But since $\alpha$ is a constant, we simplify by absorbing $\frac{1}{m}$ into the learning rate and just call the whole thing $\alpha$.

### Logistic Regression Algorithm

Here are our steps for logistic regression:

1. Start with random weights: $w_1, \ldots, w_n, b$
2. For every point $(x_1, \ldots, x_n)$:
    - For $i = 1 \ldots n$:
      - Update $w_i' \leftarrow w_i + \alpha (y - \hat{y}) x_i$
    - Update $b' \leftarrow b + \alpha (y - \hat{y})$
3. Repeat until the error is small
